# 📊 Task 2 — Exploratory Data Analysis (EDA)
**Data Science Internship at SWYNEX Technologies**  
**Author:** Affan Inamdar  
**Dataset:** Cleaned Sales Dataset (from Task 1)

In [ ]:
# ── STEP 1: Import Libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

plt.rcParams['figure.figsize']   = (14, 6)
plt.rcParams['axes.titlesize']   = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['figure.dpi']       = 100

ORANGE = '#f97316'
BLUE   = '#3b82f6'
GREEN  = '#22c55e'
RED    = '#ef4444'
PURPLE = '#8b5cf6'
COLORS = [ORANGE, BLUE, GREEN, RED, PURPLE, '#facc15', '#14b8a6', '#ec4899']

# ── HARDCODED SCREENSHOTS FOLDER — your exact path ──
SAVE_FOLDER = r'C:\Users\VICTUS\OneDrive\Desktop\data-cleaning-task\Screenshots'

# Verify and create folder
if not os.path.exists(SAVE_FOLDER):
    try:
        os.makedirs(SAVE_FOLDER, exist_ok=True)
        print(f'Created folder: {SAVE_FOLDER}')
    except Exception as e:
        # If path fails, save next to notebook
        SAVE_FOLDER = os.path.join(os.getcwd(), 'Screenshots')
        os.makedirs(SAVE_FOLDER, exist_ok=True)
        print(f'Using fallback folder: {SAVE_FOLDER}')
else:
    print(f'Folder found: {SAVE_FOLDER}')

pd.set_option('display.max_columns', None)

print('\nStep 1 Done — Libraries imported')
print(f'All charts will save to: {SAVE_FOLDER}')
print(f'Folder exists: {os.path.exists(SAVE_FOLDER)}')

In [ ]:
# ── STEP 2: Load Cleaned Dataset ──
# Try multiple locations for cleaned_data.csv
csv_candidates = [
    'cleaned_data.csv',
    os.path.join(os.getcwd(), 'cleaned_data.csv'),
    os.path.join(os.path.dirname(SAVE_FOLDER), 'cleaned_data.csv'),
]
df = None
for path in csv_candidates:
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='latin1')
        print(f'✅ Loaded from: {path}')
        break
if df is None:
    raise FileNotFoundError('cleaned_data.csv not found. Make sure it is in the same folder as this notebook.')

print(f'📊 Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── STEP 3: Identify Key Columns Automatically ──
rev_col = None
for c in ['Final_Sale_Amount','Net_Sale_Amount','Total_Compensation',
          'Sales','Revenue','Amount','Order_Value','Salary']:
    if c in df.columns:
        rev_col = c; break
if rev_col is None:
    rev_col = df.select_dtypes(include='number').columns[0]

cat_col = None
for c in ['Product','Category','Product_Category','Department','Item']:
    if c in df.columns:
        cat_col = c; break

city_col = None
for c in ['City','Region','Location','Area']:
    if c in df.columns:
        city_col = c; break

date_col = None
for c in df.columns:
    if 'date' in c.lower():
        date_col = c; break

pay_col    = next((c for c in df.columns if 'payment' in c.lower()), None)
status_col = next((c for c in df.columns if 'status'  in c.lower()), None)

print(f'Revenue column  : {rev_col}')
print(f'Category column : {cat_col}')
print(f'City column     : {city_col}')
print(f'Date column     : {date_col}')
print(f'Payment column  : {pay_col}')
print(f'Status column   : {status_col}')

if date_col:
    df[date_col]   = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    df['_Year']    = df[date_col].dt.year
    df['_Month']   = df[date_col].dt.month
    df['_MonthYr'] = df[date_col].dt.to_period('M').astype(str)
    df['_Quarter'] = 'Q' + df[date_col].dt.quarter.astype(str)
    print('Date column parsed successfully')

num_cols = [c for c in df.select_dtypes(include='number').columns if not c.startswith('_')]
print(f'Numeric columns : {num_cols}')
print('\nStep 3 Done — Columns identified')

In [ ]:
# ── STEP 4: Descriptive Statistics ──
print('=== DESCRIPTIVE STATISTICS ===')
print(df.describe(include='number').T.to_string())
print('\nStep 4 Done')

In [ ]:
# ── CHART 1: Distribution of Revenue ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'CHART 1 - Distribution of {rev_col}', fontsize=14, fontweight='bold')

axes[0].hist(df[rev_col].dropna(), bins=40, color=ORANGE, edgecolor='white', alpha=0.85)
axes[0].axvline(df[rev_col].mean(),   color='black', linestyle='--', linewidth=2,
                label=f'Mean={df[rev_col].mean():,.0f}')
axes[0].axvline(df[rev_col].median(), color=RED,     linestyle='-',  linewidth=2,
                label=f'Median={df[rev_col].median():,.0f}')
axes[0].set_title(f'Histogram of {rev_col}'); axes[0].set_xlabel(rev_col)
axes[0].set_ylabel('Frequency'); axes[0].legend()

bp = axes[1].boxplot(df[rev_col].dropna(), patch_artist=True)
bp['boxes'][0].set_facecolor(ORANGE); bp['boxes'][0].set_alpha(0.7)
axes[1].set_title(f'Box Plot of {rev_col}'); axes[1].set_ylabel(rev_col)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart1_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart 1 saved to: {SAVE_FOLDER}')
skew = df[rev_col].skew()
print(f'INSIGHT 1: Mean={df[rev_col].mean():,.2f} | Median={df[rev_col].median():,.2f} | Skewness={skew:.3f}')
print(f'   -> {"Right-skewed: most values below average" if skew>0 else "Left-skewed"}')

In [ ]:
# ── CHART 2: Revenue by City/Region ──
if city_col:
    city_data = df.groupby(city_col)[rev_col].sum().sort_values(ascending=False).reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'CHART 2 - Revenue by {city_col}', fontsize=14, fontweight='bold')

    bars = axes[0].bar(city_data[city_col], city_data[rev_col],
                       color=COLORS[:len(city_data)], edgecolor='white')
    axes[0].set_title(f'Total Revenue by {city_col}')
    axes[0].set_xlabel(city_col); axes[0].set_ylabel('Total Revenue')
    axes[0].tick_params(axis='x', rotation=45)
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x()+bar.get_width()/2, h*1.01,
                     f'{h:,.0f}', ha='center', fontsize=8, fontweight='bold')

    axes[1].pie(city_data[rev_col], labels=city_data[city_col], autopct='%1.1f%%',
                colors=COLORS[:len(city_data)], startangle=140,
                wedgeprops={'edgecolor':'white','linewidth':1.5})
    axes[1].set_title(f'Revenue Share by {city_col}')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart2_revenue_by_city.png'), dpi=150, bbox_inches='tight')
    plt.show()
    top = city_data.iloc[0]; bot = city_data.iloc[-1]
    print(f'Chart 2 saved to: {SAVE_FOLDER}')
    print(f'INSIGHT 2: Highest = {top[city_col]} ({top[rev_col]:,.0f}) | Lowest = {bot[city_col]} ({bot[rev_col]:,.0f})')
else:
    print('No city/region column found')

In [ ]:
# ── CHART 3: Revenue by Category ──
if cat_col:
    cat_data = df.groupby(cat_col)[rev_col].agg(['sum','mean','count'])\
                 .rename(columns={'sum':'Total','mean':'Average','count':'Count'})\
                 .sort_values('Total', ascending=False).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'CHART 3 - Revenue by {cat_col}', fontsize=14, fontweight='bold')

    top10 = cat_data.head(10)
    axes[0].barh(top10[cat_col], top10['Total'],
                 color=COLORS[:len(top10)], edgecolor='white')
    axes[0].set_title(f'Top {cat_col} by Total Revenue')
    axes[0].set_xlabel('Total Revenue'); axes[0].invert_yaxis()

    axes[1].bar(cat_data[cat_col], cat_data['Average'],
                color=COLORS[:len(cat_data)], edgecolor='white')
    axes[1].set_title(f'Average Revenue per {cat_col}')
    axes[1].set_xlabel(cat_col); axes[1].set_ylabel('Average Revenue')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart3_revenue_by_category.png'), dpi=150, bbox_inches='tight')
    plt.show()
    top_c = cat_data.iloc[0]
    print(f'Chart 3 saved to: {SAVE_FOLDER}')
    print(f'INSIGHT 3: Best {cat_col} = {top_c[cat_col]} | Total={top_c["Total"]:,.0f} | Avg={top_c["Average"]:,.0f}')
    print(cat_data.to_string(index=False))
else:
    print('No category column found')

In [ ]:
# ── CHART 4: Sales Trend Over Time ──
if date_col and '_MonthYr' in df.columns:
    monthly   = df.groupby('_MonthYr')[rev_col].sum().reset_index().sort_values('_MonthYr')
    quarterly = df.groupby('_Quarter')[rev_col].sum().reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('CHART 4 - Sales Trend Over Time', fontsize=14, fontweight='bold')

    x = range(len(monthly))
    axes[0].plot(x, monthly[rev_col], color=ORANGE, linewidth=2.5, marker='o', markersize=7)
    axes[0].fill_between(x, monthly[rev_col], alpha=0.15, color=ORANGE)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(monthly['_MonthYr'], rotation=45, ha='right', fontsize=8)
    axes[0].set_title('Monthly Revenue Trend'); axes[0].set_ylabel('Revenue')

    axes[1].bar(quarterly['_Quarter'], quarterly[rev_col],
                color=[ORANGE,BLUE,GREEN,RED][:len(quarterly)],
                edgecolor='white', width=0.5)
    axes[1].set_title('Revenue by Quarter')
    axes[1].set_xlabel('Quarter'); axes[1].set_ylabel('Revenue')
    for i, row in quarterly.iterrows():
        axes[1].text(i, row[rev_col]*1.01, f"{row[rev_col]:,.0f}",
                     ha='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart4_sales_trend.png'), dpi=150, bbox_inches='tight')
    plt.show()
    best_m = monthly.loc[monthly[rev_col].idxmax()]
    print(f'Chart 4 saved to: {SAVE_FOLDER}')
    print(f'INSIGHT 4: Peak month = {best_m["_MonthYr"]} ({best_m[rev_col]:,.0f})')
    print(f'           Best quarter = {quarterly.loc[quarterly[rev_col].idxmax(),"_Quarter"]}')
else:
    print('No date column found - skipping trend chart')

In [ ]:
# ── CHART 5: Correlation Analysis ──
corr = df[num_cols].corr()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('CHART 5 - Correlation Analysis', fontsize=14, fontweight='bold')

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=axes[0], annot_kws={'size':8})
axes[0].set_title('Correlation Matrix (Lower Triangle)')

if rev_col in corr.columns:
    rev_corr   = corr[rev_col].drop(rev_col).sort_values(ascending=False)
    bar_colors = [GREEN if v > 0 else RED for v in rev_corr.values]
    axes[1].barh(rev_corr.index, rev_corr.values, color=bar_colors, edgecolor='white')
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title(f'Correlation with {rev_col}')
    axes[1].set_xlabel('Correlation Coefficient')
    for i, (idx, val) in enumerate(rev_corr.items()):
        axes[1].text(val+(0.01 if val>=0 else -0.01), i,
                     f'{val:.3f}', va='center',
                     ha='left' if val>=0 else 'right', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart5_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart 5 saved to: {SAVE_FOLDER}')
if rev_col in corr.columns:
    strongest = rev_corr.abs().idxmax()
    print(f'INSIGHT 5: Strongest correlation with {rev_col} = "{strongest}" (r={rev_corr[strongest]:.3f})')

In [ ]:
# ── CHART 6: Outlier Detection ──
plot_num = num_cols[:6]
n = len(plot_num); cols_r = 3; rows_r = (n + cols_r - 1) // cols_r
fig, axes = plt.subplots(rows_r, cols_r, figsize=(18, rows_r*5))
axes_f = axes.flatten() if n > 1 else [axes]
fig.suptitle('CHART 6 - Outlier Detection (Box Plots)', fontsize=14, fontweight='bold')

outlier_info = []
for i, col in enumerate(plot_num):
    Q1=df[col].quantile(0.25); Q3=df[col].quantile(0.75); IQR=Q3-Q1
    n_out = ((df[col]<Q1-1.5*IQR)|(df[col]>Q3+1.5*IQR)).sum()
    outlier_info.append({'Column':col,'Outliers':n_out,'Pct%':round(n_out/len(df)*100,1)})
    bp = axes_f[i].boxplot(df[col].dropna(), patch_artist=True)
    bp['boxes'][0].set_facecolor(COLORS[i%len(COLORS)]); bp['boxes'][0].set_alpha(0.75)
    axes_f[i].set_title(f'{col} ({n_out} outliers | {n_out/len(df)*100:.1f}%)')
    axes_f[i].set_ylabel('Value')

for j in range(i+1, len(axes_f)):
    axes_f[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart6_outliers.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart 6 saved to: {SAVE_FOLDER}')
print('INSIGHT 6 - Outlier Summary:')
print(pd.DataFrame(outlier_info).to_string(index=False))

In [ ]:
# ── CHART 7: Payment, Status & Demographics ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('CHART 7 - Payment, Status & Demographics', fontsize=14, fontweight='bold')

if pay_col and pay_col in df.columns:
    pc = df[pay_col].value_counts()
    axes[0].bar(pc.index, pc.values, color=COLORS[:len(pc)], edgecolor='white')
    axes[0].set_title(f'{pay_col} Distribution'); axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    for i,v in enumerate(pc.values):
        axes[0].text(i, v*1.01, str(v), ha='center', fontsize=9, fontweight='bold')
else:
    axes[0].text(0.5,0.5,'No Payment column',ha='center',va='center',transform=axes[0].transAxes)

if status_col and status_col in df.columns:
    sc = df[status_col].value_counts()
    axes[1].pie(sc.values, labels=sc.index, autopct='%1.1f%%',
                colors=COLORS[:len(sc)], startangle=90,
                wedgeprops={'edgecolor':'white','linewidth':1.5})
    axes[1].set_title(f'{status_col} Breakdown')
else:
    axes[1].text(0.5,0.5,'No Status column',ha='center',va='center',transform=axes[1].transAxes)

gen_col = next((c for c in df.columns if 'gender' in c.lower() or 'sex' in c.lower()), None)
rat_col = next((c for c in df.columns if 'rating' in c.lower() or 'score' in c.lower()), None)
if gen_col:
    gc = df[gen_col].value_counts()
    axes[2].bar(gc.index, gc.values, color=[BLUE,ORANGE], edgecolor='white', width=0.5)
    axes[2].set_title(f'{gen_col} Distribution'); axes[2].set_ylabel('Count')
    for i,v in enumerate(gc.values):
        axes[2].text(i, v*1.01, str(v), ha='center', fontsize=10, fontweight='bold')
elif rat_col:
    axes[2].hist(df[rat_col].dropna(), bins=20, color=PURPLE, edgecolor='white')
    axes[2].set_title(f'{rat_col} Distribution'); axes[2].set_xlabel(rat_col)
else:
    axes[2].text(0.5,0.5,'No Gender/Rating column',ha='center',va='center',transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart7_payment_status.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart 7 saved to: {SAVE_FOLDER}')
if pay_col and pay_col in df.columns:
    pc2 = df[pay_col].value_counts()
    print(f'INSIGHT 7: Most popular payment = {pc2.index[0]} ({pc2.values[0]} orders, {pc2.values[0]/len(df)*100:.1f}%)')

In [ ]:
# ── CHART 8: Final EDA Summary Dashboard ──
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#f8fafc')

fig.text(0.5, 0.97, 'EDA Summary Dashboard - Sales Dataset',
         ha='center', fontsize=20, fontweight='bold', color='#1e293b')
fig.text(0.5, 0.94,
         f'Records: {len(df):,} | Columns: {df.shape[1]} | '
         f'Total {rev_col}: {df[rev_col].sum():,.0f} | '
         f'Author: Affan Inamdar | SWYNEX Task 2',
         ha='center', fontsize=10, color='#64748b')

kpis = [
    ('Total Records', f'{len(df):,}',                  ORANGE),
    ('Total Revenue', f'{df[rev_col].sum():,.0f}',     GREEN),
    ('Avg Revenue',   f'{df[rev_col].mean():,.0f}',    BLUE),
    ('Max Revenue',   f'{df[rev_col].max():,.0f}',     PURPLE),
    ('Unique Cat.',   str(df[cat_col].nunique()) if cat_col else 'N/A', RED),
]
for i,(lbl,val,col) in enumerate(kpis):
    ax=fig.add_axes([0.02+i*0.196, 0.82, 0.17, 0.09])
    ax.set_facecolor(col)
    ax.text(0.5,0.63,val,ha='center',va='center',fontsize=15,fontweight='bold',color='white',transform=ax.transAxes)
    ax.text(0.5,0.22,lbl,ha='center',va='center',fontsize=9,color='white',alpha=0.9,transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)

ax1=fig.add_axes([0.02,0.44,0.44,0.34])
if cat_col:
    top8=df.groupby(cat_col)[rev_col].sum().sort_values(ascending=True).tail(8)
    ax1.barh(top8.index,top8.values,color=COLORS[:len(top8)],edgecolor='white')
    ax1.set_title('Top Categories by Revenue',fontweight='bold'); ax1.set_xlabel('Revenue')

ax2=fig.add_axes([0.54,0.44,0.44,0.34])
if '_MonthYr' in df.columns:
    mon2=df.groupby('_MonthYr')[rev_col].sum().reset_index().sort_values('_MonthYr')
    ax2.plot(range(len(mon2)),mon2[rev_col],color=ORANGE,linewidth=2.5,marker='o',markersize=6)
    ax2.fill_between(range(len(mon2)),mon2[rev_col],alpha=0.15,color=ORANGE)
    ax2.set_xticks(range(len(mon2)))
    ax2.set_xticklabels(mon2['_MonthYr'],rotation=45,ha='right',fontsize=7)
    ax2.set_title('Monthly Revenue Trend',fontweight='bold'); ax2.set_ylabel('Revenue')
else:
    ax2.text(0.5,0.5,'No date data',ha='center',va='center',transform=ax2.transAxes)

ax3=fig.add_axes([0.02,0.04,0.29,0.34])
cols6=[c for c in num_cols if not c.startswith('_')][:6]
if len(cols6)>=2:
    sns.heatmap(df[cols6].corr(),annot=True,fmt='.1f',cmap='RdYlGn',
                center=0,ax=ax3,cbar=False,linewidths=0.5,annot_kws={'size':7})
    ax3.set_title('Correlation Matrix',fontweight='bold')

ax4=fig.add_axes([0.36,0.04,0.29,0.34])
if city_col:
    cr2=df.groupby(city_col)[rev_col].sum().sort_values(ascending=False)
    ax4.bar(cr2.index,cr2.values,color=COLORS[:len(cr2)],edgecolor='white')
    ax4.set_title(f'Revenue by {city_col}',fontweight='bold'); ax4.set_ylabel('Revenue')
    ax4.tick_params(axis='x',rotation=45)

ax5=fig.add_axes([0.70,0.04,0.28,0.34])
ax5.hist(df[rev_col].dropna(),bins=35,color=PURPLE,edgecolor='white',alpha=0.8)
ax5.axvline(df[rev_col].mean(),color=RED,linestyle='--',linewidth=2,label=f'Mean={df[rev_col].mean():,.0f}')
ax5.set_title('Revenue Distribution',fontweight='bold')
ax5.set_xlabel('Revenue'); ax5.set_ylabel('Frequency'); ax5.legend(fontsize=8)

plt.savefig(os.path.join(SAVE_FOLDER, 'eda_chart8_summary_dashboard.png'),
            dpi=150, bbox_inches='tight', facecolor='#f8fafc')
plt.show()
print(f'Chart 8 (Summary Dashboard) saved to: {SAVE_FOLDER}')
print('USE chart8 AS YOUR LINKEDIN THUMBNAIL')

In [ ]:
# ── FINAL SUMMARY ──
print('=' * 60)
print('   EDA COMPLETE - TASK 2 SUMMARY REPORT')
print('   SWYNEX Technologies | Affan Inamdar')
print('=' * 60)
print(f'DATASET : {len(df):,} rows x {df.shape[1]} columns')
print(f'REVENUE COLUMN : {rev_col}')
print(f'  Total  : {df[rev_col].sum():>15,.2f}')
print(f'  Mean   : {df[rev_col].mean():>15,.2f}')
print(f'  Median : {df[rev_col].median():>15,.2f}')
print(f'  Std    : {df[rev_col].std():>15,.2f}')
print(f'  Min    : {df[rev_col].min():>15,.2f}')
print(f'  Max    : {df[rev_col].max():>15,.2f}')
print()
print('CHARTS SAVED TO:', SAVE_FOLDER)
saved = [f for f in os.listdir(SAVE_FOLDER) if f.startswith('eda_chart') and f.endswith('.png')]
for f in sorted(saved):
    print(f'  {f}')
print()
print('Task 2 EDA Complete!')